In [1]:
# Configurar el path para importar los módulos
import sys
import os

# Añadir el directorio raíz del proyecto al path
root_dir = os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..')))
if root_dir not in sys.path:
    sys.path.append(root_dir)

In [2]:
# Importar las funciones necesarias
from db.conexion_db import crear_conexion, cerrar_conexion
from db.querys_db import insert_data_pandas, consult_data, update_data
import datetime
import pandas as pd
import numpy as np

In [3]:
columnas_materiales = {
    "Material": "material",
	"Texto breve de material" : "descripcion",
    "Tipo material": "codigo_tipo_material",
    "Grupo de artículos": "codigo_grupo_articulo",
    "Nº antiguo material": "codigo_antiguo",
    "Unidad medida base": "unidad_medida",
    "Peso bruto":"peso_bruto",
    "Peso neto":"peso_neto",
    "Unidad de peso": "unidad_peso",
    "Volumen":"volumen",
    "Unidad de volumen": "unidad_volumen",
    "Jerarquía productos":"jerarquia",
    "Nº pieza fabricante":"numero_pieza",
    "Creado el":"creado_el",
    "Creado por":"creado_por",
    "Código EAN/UPC":"codigo_ean"

}

In [4]:
df_materiales = pd.read_excel(r'C:\Users\CGM\Projects\Proyecto CGMExPress\maestros\materiales.xlsx')

In [5]:
connection = crear_conexion("db_matricial")
db_materiales = consult_data(connection, 'SELECT id_material,codigo_material FROM tbl_materiales_ih09')
cerrar_conexion(connection)

Conexion Exitosa
conexion cerrada


In [6]:
connection = crear_conexion("db_matricial")
db_tipo_material = consult_data(connection, 'SELECT id_tipo_material,codigo_tipo_material FROM tbl_tipo_material')
cerrar_conexion(connection)

Conexion Exitosa
conexion cerrada


In [7]:
connection = crear_conexion("db_matricial")
db_grupo_articulo = consult_data(connection, 'SELECT id_grupo_articulo,codigo_grupo_articulo FROM tbl_grupo_articulo')
cerrar_conexion(connection)

Conexion Exitosa
conexion cerrada


In [8]:
df_materiales.rename(columns=columnas_materiales, inplace=True)
df_material = pd.DataFrame(db_materiales)
df_grupo_articulo = pd.DataFrame(db_grupo_articulo)
df_tipo_material = pd.DataFrame(db_tipo_material)

In [9]:
df_materiales_no_encontrados = df_materiales[
    ~df_materiales['material'].isin(df_material['codigo_material'])
]

In [10]:
df_materiales_no_encontrados

,material,descripcion,codigo_tipo_material,codigo_grupo_articulo,codigo_antiguo,unidad_medida,peso_bruto,peso_neto,unidad_peso,volumen,unidad_volumen,jerarquia,numero_pieza,creado_el,creado_por,codigo_ean
0,924402496,PASADOR,ZMER,G038,0924402496,UND,0.01,0.01,KG,0.01,M3,0101E010103,9.24E+08,2026-01-02,OTABOADA,924402496
1,4018000001,ASIENTO\RE232899,ZMER,G016,RE232899,UND,0.01,0.01,KG,0.01,M3,0101C010023,RE232899,2025-11-06,CPEREZ,4018000001
4,6050000012,"LIJADORA ORBITAL 6"" PREMIUM",ZMER,G053,6050000012,UND,0.01,0.01,KG,0.01,M3,0101H010143,6050000012,2025-11-11,CPEREZ,6050000012
5,6050000015,"CUADRO CUDRADO 1""",ZMER,G053,6050000015,UND,0.01,0.01,KG,0.01,M3,0101H010143,6050000015,2025-12-15,CPEREZ,6050000015
6,6050000016,CANCAMO METRICO M36 4.5T,ZMER,G053,6050000016,UND,0.01,0.01,KG,0.01,M3,0310H020367,6050000016,2025-12-15,CPEREZ,6050000016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3252,TETC2500/5,TRANS.CORCL 05 15VA 3KVTC 2500/5 50-60HZ,ZMER,G025,TC2500/5,UND,0.01,0.01,KG,0.01,M3,0101C040067,TC2500/5,2025-09-23,JTUDELANO,TC2500/5
3308,UNI0000083,"UNIFORME ESTAND.DRIL AZUL,TALLA XXL,S/LO",ZMER,G057,UNI0000083,UND,0.01,0.01,KG,0.01,M3,0310I020370,NaN,2024-05-03,NMOORE,NaN
3311,UNI0000140\GEN,"POLO MANGA LARGA AZUL, TALLA XXXL,S/LOGO",ZMER,G057,UNI0000140,UND,0.01,0.01,KG,0.01,M3,0310I020370,UNI0000140,2024-06-28,NMOORE,UNI0000140
3312,UNI0000142,CHALECO ROJO - L,ZMER,G057,UNI0000142,UND,0.01,0.01,KG,0.01,M3,0310I020370,UNI0000142,2025-01-06,NMOORE,UNI0000142


In [11]:
df_join_group = df_materiales_no_encontrados.merge(df_grupo_articulo, on="codigo_grupo_articulo", how='left')
df_join = df_join_group.merge(df_tipo_material, on="codigo_tipo_material", how='left')

In [12]:
df_normalized = df_join[['material','descripcion','id_tipo_material','id_grupo_articulo','codigo_antiguo',
                         'unidad_medida','peso_bruto','peso_neto','unidad_peso','volumen','unidad_volumen',
                         'jerarquia','numero_pieza','codigo_ean',"creado_por","creado_el"]]

In [13]:
cabezera_materiales = ['codigo_material','descripcion','id_tipo_material','id_grupo_articulo','codigo_antiguo',
                       'unidad_medida_base','peso_bruto','peso_neto','unidad_peso','volumen','unidad_volumen',
                       'jerarquia','numero_pieza','codigo_ean','created_by_sap','created_at_sap']

In [14]:
if not df_materiales.empty:
    connection = crear_conexion("db_matricial")
    insert_data_pandas(connection,'tbl_materiales_ih09',df_normalized,cabezera_materiales,200)
else:   
    print('Dataframe en blanco')

Conexion Exitosa
 Se insertaron: 936 registros en la tabla tbl_materiales_ih09
conexion cerrada
